In [51]:
import pandas as pd

df = pd.read_pickle('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/01_cleaned_master_data.pkl')

#print(df.info())
#print(df.head(5))
#print(df['departure'].unique())
print(df['date'].unique())

<DatetimeArray>
['2005-07-02 00:00:00', '2005-07-03 00:00:00', '2005-07-04 00:00:00', '2005-07-05 00:00:00', '2005-07-06 00:00:00', '2005-07-07 00:00:00', '2005-07-08 00:00:00', '2005-07-09 00:00:00', '2005-07-10 00:00:00', '2005-07-12 00:00:00',
 ...
 '2025-09-04 00:00:00', '2025-09-05 00:00:00', '2025-09-06 00:00:00', '2025-09-07 00:00:00', '2025-09-09 00:00:00', '2025-09-10 00:00:00', '2025-09-11 00:00:00', '2025-09-12 00:00:00', '2025-09-13 00:00:00', '2025-09-14 00:00:00']
Length: 1306, dtype: datetime64[ns]


In [35]:
import requests
import json

# API-URL
url = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=52.52"
    "&longitude=13.41"
    "&start_date=2026-04-02"
    "&end_date=2026-04-05"
    "&daily=temperature_2m_mean"
    "&hourly=temperature_2m,rain,wind_speed_10m"
    "&timezone=Europe%2FBerlin"
)

# API-Anfrage senden
response = requests.get(url)

# Statuscode prüfen
if response.status_code == 200:
    data = response.json()

    # --- Allgemeine Infos ---
    print("=" * 50)
    print(f"Ort:       Breite {data['latitude']}°, Länge {data['longitude']}°")
    print(f"Zeitzone:  {data['timezone']} (UTC{data['utc_offset_seconds']//3600:+d}h)")
    print(f"Zeitraum:  {data['daily']['time'][0]} bis {data['daily']['time'][-1]}")
    print("=" * 50)

    # --- Tagesdaten: Mittlere Temperatur ---
    print("\n📅 TAGESDATEN – Mittlere Temperatur (2m)")
    print(f"{'Datum':<15} {'Ø Temperatur (°C)':>20}")
    print("-" * 37)
    daily = data["daily"]
    for date, temp in zip(daily["time"], daily["temperature_2m_mean"]):
        temp_str = f"{temp:.1f}" if temp is not None else "N/A"
        print(f"{date:<15} {temp_str:>20}")

    # --- Stundendaten ---
    print("\n⏱️  STUNDENDATEN (erste 24 Einträge)")
    print(f"{'Zeitstempel':<22} {'Temp (°C)':>12} {'Regen (mm)':>12} {'Wind (km/h)':>12}")
    print("-" * 60)
    hourly = data["hourly"]
    for i, (ts, temp, rain, wind) in enumerate(zip(
        hourly["time"],
        hourly["temperature_2m"],
        hourly["rain"],
        hourly["wind_speed_10m"]
    )):
        if i >= 24:
            break
        temp_str = f"{temp:.1f}" if temp is not None else "N/A"
        rain_str = f"{rain:.2f}" if rain is not None else "N/A"
        wind_str = f"{wind:.1f}" if wind is not None else "N/A"
        print(f"{ts:<22} {temp_str:>12} {rain_str:>12} {wind_str:>12}")

else:
    print(f"Fehler beim Abrufen der Daten: HTTP {response.status_code}")
    print(response.text)

Ort:       Breite 52.54833°, Länge 13.407822°
Zeitzone:  Europe/Berlin (UTC+2h)
Zeitraum:  2026-04-02 bis 2026-04-05

📅 TAGESDATEN – Mittlere Temperatur (2m)
Datum              Ø Temperatur (°C)
-------------------------------------
2026-04-02                       6.2
2026-04-03                       7.1
2026-04-04                      11.0
2026-04-05                      11.7

⏱️  STUNDENDATEN (erste 24 Einträge)
Zeitstempel               Temp (°C)   Regen (mm)  Wind (km/h)
------------------------------------------------------------
2026-04-02T00:00                3.8         0.00          4.6
2026-04-02T01:00                2.8         0.00          4.3
2026-04-02T02:00                2.1         0.00          4.2
2026-04-02T03:00                1.4         0.00          4.5
2026-04-02T04:00                1.1         0.00          4.7
2026-04-02T05:00                0.6         0.00          4.7
2026-04-02T06:00                0.4         0.00          2.9
2026-04-02T07:00        

In [ ]:
import pickle
import requests
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# --- 1. Pickle laden ---
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/01_cleaned_master_data.pkl', "rb") as f:
    df = pickle.load(f)

# --- 2. Geocoding einrichten ---
geolocator = Nominatim(user_agent="mein_wetter_projekt")            # weiss noch nicht was diese Zeile macht
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def get_coords(ortsname):
    try:
        loc = geocode(str(ortsname))
        if loc:
            return loc.latitude, loc.longitude
    except Exception:
        pass
    return None, None

# --- 3. Koordinaten für departure und arrival ermitteln ---
# Alle einzigartigen Orte sammeln (spart API-Aufrufe!)
alle_orte = pd.unique(df[["departure", "arrival"]].values.ravel())

coords_cache = {}
for ort in alle_orte:
    if pd.notna(ort):
        coords_cache[ort] = get_coords(ort)
        print(f"📍 {ort}: {coords_cache[ort]}")

# --- 4. Koordinaten in neue Spalten eintragen ---
df["departure_lat"] = df["departure"].map(lambda o: coords_cache.get(o, (None, None))[0])
df["departure_lon"] = df["departure"].map(lambda o: coords_cache.get(o, (None, None))[1])
df["arrival_lat"]   = df["arrival"].map(lambda o: coords_cache.get(o, (None, None))[0])
df["arrival_lon"]   = df["arrival"].map(lambda o: coords_cache.get(o, (None, None))[1])

# --- 5. Speichern ---
df.to_pickle("cleaned_master_data_with_coordinates.pkl")
#df.to_csv("cleaned_master_data_with_coordinates.csv", index=False)

print(df[["departure", "departure_lat", "departure_lon",
          "arrival",   "arrival_lat",   "arrival_lon"]].head())

📍 Fromentine: (46.8896706, -2.1366623)
📍 Noirmoutier-en-l'Ile: (47.000089, -2.2441449)
📍 Challans: (46.8478091, -1.877431)
📍 Les Essarts: (48.880997, 0.9921208)
📍 La Châtaigneraie: (46.65, -0.7391667)
📍 Tours: (47.3900474, 0.6889268)
📍 Blois: (47.5876861, 1.3337639)
📍 Chambord: (48.8899454, 0.6095948)
📍 Montargis: (47.9978628, 2.7310072)
📍 Troyes: (48.2971626, 4.0746257)
📍 Nancy: (48.6937223, 6.1834097)
📍 Lunéville: (48.5916164, 6.4919563)
📍 Karlsruhe: (49.0068705, 8.4034195)
📍 Pforzheim: (48.890934, 8.7025509)
📍 Gérardmer: (48.0705772, 6.8776965)
📍 Mulhouse: (47.7467233, 7.3389937)
📍 Brignoud: (45.2625277, 5.9013413)
📍 Courchevel - Altiport: (45.3971612, 6.6334623)
📍 Courchevel: (45.4140984, 6.6349892)
📍 Briançon: (44.8984037, 6.6436313)
📍 Digne-les-Bains: (44.0917585, 6.2346362)
📍 Miramas: (43.582733, 5.0025809)
📍 Montpellier: (43.6112422, 3.8767337)
📍 Agde: (43.3124696, 3.4707789)
📍 Ax-3 Domaines: (42.7005767, 1.8152092)
📍 Lézat-sur-Leze: (43.2773, 1.34623)
📍 Saint-Lary-Soulan (Pla 

gfgfdgfdf

In [54]:
import pickle
import requests
import pandas as pd

# --- 1. Pickle mit Koordinaten laden ---
with open("/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/cleaned_master_data_with_coordinates.pkl", "rb") as f:
    df = pickle.load(f)

# --- 2. Wetterdaten-Funktion (hourly, alle Variablen) ---
def get_wetter(lat, lon, start, end):
    if pd.isna(lat) or pd.isna(lon):
        return {}
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={start}&end_date={end}"
        f"&hourly=temperature_2m,rain,wind_speed_10m,"
        f"relativehumidity_2m,precipitation,winddirection_10m"
        f"&timezone=Europe%2FBerlin"
    )
    response = requests.get(url)
    if response.status_code == 200:
        return response.json().get("hourly", {})
    return {}

# --- 3. Cache ---
wetter_cache = {}

def wetter_fuer_ort(lat, lon, start, end):
    key = (round(lat, 4), round(lon, 4))
    if key not in wetter_cache:
        wetter_cache[key] = get_wetter(lat, lon, start, end)
    return wetter_cache[key]

# --- 4. Hilfsfunktion: Mittelwert für bestimmte Stunden eines Datums ---
def mittelwert_stundenbereich(daten, datum, stunden_von, stunden_bis, key):
    zeiten = daten.get("time", [])
    werte  = daten.get(key, [])

    gefiltert = [
        wert for zeit, wert in zip(zeiten, werte)
        if zeit.startswith(datum)
        and stunden_von <= int(zeit[11:13]) <= stunden_bis
        and wert is not None
    ]

    return round(sum(gefiltert) / len(gefiltert), 2) if gefiltert else None

# --- 5. Datumsspanne automatisch ermitteln ---
df["date"] = pd.to_datetime(df["date"])

START = df["date"].min().strftime("%Y-%m-%d")
END   = df["date"].max().strftime("%Y-%m-%d")

print(f"📅 Datumsspanne: {START} bis {END}")

# --- 6. Wettervariablen definieren: API-Name → Spaltenname ---
wetter_variablen = {
    "temperature_2m":       "temp_mittel",
    "rain":                 "regen_mittel",
    "wind_speed_10m":       "wind_mittel",
    "relativehumidity_2m":  "luftfeuchte_mittel",
    "precipitation":        "niederschlag_mittel",
    "winddirection_10m":    "windrichtung_mittel",
}

# --- 7. Stundenbereiche je Prefix definieren (jeweils inklusiv) ---
stunden_config = {
    "departure": (12, 15),  # 12:00 – 15:00 Uhr (inkl.)
    "arrival":   (15, 18),  # 15:00 – 18:00 Uhr (inkl.)
}

# --- 8. Wetterwerte pro Zeile eintragen ---
for prefix, (std_von, std_bis) in stunden_config.items():
    lat_col = f"{prefix}_lat"
    lon_col = f"{prefix}_lon"

    # Eine Liste pro Variable
    listen = {api_key: [] for api_key in wetter_variablen}

    for _, row in df.iterrows():
        lat, lon = row[lat_col], row[lon_col]
        datum = row["date"].strftime("%Y-%m-%d")

        if pd.isna(lat) or pd.isna(lon):
            for api_key in wetter_variablen:
                listen[api_key].append(None)
            continue

        daten = wetter_fuer_ort(lat, lon, START, END)

        for api_key in wetter_variablen:
            listen[api_key].append(
                mittelwert_stundenbereich(daten, datum, std_von, std_bis, api_key)
            )

    # Spalten in DataFrame eintragen
    for api_key, spalten_suffix in wetter_variablen.items():
        df[f"{prefix}_{spalten_suffix}"] = listen[api_key]

    print(f"✅ Wetterdaten für '{prefix}' ({std_von}:00–{std_bis}:00 Uhr inkl.) hinzugefügt")

# --- 9. Speichern ---
df.to_pickle("/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/cleaned_master_data_with_coordinates_and_weather.pkl")

print(df[[
    "departure", "departure_temp_mittel", "departure_regen_mittel",
    "departure_wind_mittel", "departure_luftfeuchte_mittel",
    "departure_niederschlag_mittel", "departure_windrichtung_mittel",
    "arrival", "arrival_temp_mittel", "arrival_regen_mittel",
    "arrival_wind_mittel", "arrival_luftfeuchte_mittel",
    "arrival_niederschlag_mittel", "arrival_windrichtung_mittel"
]].head())

📅 Datumsspanne: 2005-05-08 bis 2025-09-14
✅ Wetterdaten für 'departure' (12:00–15:00 Uhr inkl.) hinzugefügt
✅ Wetterdaten für 'arrival' (15:00–18:00 Uhr inkl.) hinzugefügt
    departure  departure_temp_mittel  departure_regen_mittel  departure_wind_mittel  departure_luftfeuchte_mittel  departure_niederschlag_mittel  departure_windrichtung_mittel               arrival  arrival_temp_mittel  arrival_regen_mittel  arrival_wind_mittel  arrival_luftfeuchte_mittel  arrival_niederschlag_mittel  arrival_windrichtung_mittel
0  Fromentine                   20.0                     0.0                   12.0                         75.25                            0.0                         214.25  Noirmoutier-en-l'Ile                  NaN                   NaN                  NaN                         NaN                          NaN                          NaN
1  Fromentine                   20.0                     0.0                   12.0                         75.25                   

In [58]:
print(df.tail())

                   race  year                                                url rank           rider_url time_gap   birthdate  height                                        image_url                name nationality place_of_birth                          points_per_season_history                                     season_results                                      teams_history  weight            url_name                                 grand_tour_history  departure arrival distance vertical_meters profile_score won_how avg_speed race_ranking  one_day_races      gc  time_trial  sprint  climber  hills    0  stage_nr       date  departure_lat  departure_lon  arrival_lat  arrival_lon  departure_temp_mittel  departure_regen_mittel  departure_wind_mittel  departure_luftfeuchte_mittel  departure_niederschlag_mittel  departure_windrichtung_mittel  arrival_temp_mittel  arrival_regen_mittel  arrival_wind_mittel  arrival_luftfeuchte_mittel  arrival_niederschlag_mittel  \
225687  vuelta-a-espa

In [56]:
import pickle
import pandas as pd
import requests

# --- Gespeicherte Datei laden ---
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/cleaned_master_data_with_coordinates_and_weather.pkl', "rb") as f:
    df = pickle.load(f)

# --- DIAGNOSE 1: Koordinaten vorhanden? ---
print("Fehlende Koordinaten:")
print(df[["arrival", "arrival_lat", "arrival_lon"]].isna().sum())
print()
print("Beispielwerte arrival-Koordinaten:")
print(df[["arrival", "arrival_lat", "arrival_lon"]].head(10))

# --- DIAGNOSE 2: API-Antwort prüfen ---
START = df["date"].min().strftime("%Y-%m-%d")
END   = df["date"].max().strftime("%Y-%m-%d")

def get_wetter(lat, lon, start, end):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={start}&end_date={end}"
        f"&hourly=temperature_2m,rain,wind_speed_10m,"
        f"relativehumidity_2m,precipitation,winddirection_10m"
        f"&timezone=Europe%2FBerlin"
    )
    response = requests.get(url)
    if response.status_code == 200:
        return response.json().get("hourly", {})
    return {}

test_row = df[df["arrival_lat"].notna()].iloc[0]
lat  = test_row["arrival_lat"]
lon  = test_row["arrival_lon"]
datum = test_row["date"].strftime("%Y-%m-%d")

print(f"Test-Ort: {test_row['arrival']} | {lat}, {lon} | Datum: {datum}")

test_daten = get_wetter(lat, lon, START, END)
print(f"API-Keys zurückgegeben: {list(test_daten.keys())}")
print(f"Erste 5 Zeitstempel: {test_daten.get('time', [])[:5]}")

# --- DIAGNOSE 3: Stundenfilter prüfen ---
zeiten = test_daten.get("time", [])
gefiltert = [z for z in zeiten if z.startswith(datum) and 15 <= int(z[11:13]) <= 18]
print(f"Gefilterte Stunden für {datum} (15–18 Uhr): {gefiltert}")

Fehlende Koordinaten:
arrival            0
arrival_lat    11439
arrival_lon    11439
dtype: int64

Beispielwerte arrival-Koordinaten:
                arrival  arrival_lat  arrival_lon
0  Noirmoutier-en-l'Ile    47.000089    -2.244145
1  Noirmoutier-en-l'Ile    47.000089    -2.244145
2  Noirmoutier-en-l'Ile    47.000089    -2.244145
3  Noirmoutier-en-l'Ile    47.000089    -2.244145
4  Noirmoutier-en-l'Ile    47.000089    -2.244145
5  Noirmoutier-en-l'Ile    47.000089    -2.244145
6  Noirmoutier-en-l'Ile    47.000089    -2.244145
7  Noirmoutier-en-l'Ile    47.000089    -2.244145
8  Noirmoutier-en-l'Ile    47.000089    -2.244145
9  Noirmoutier-en-l'Ile    47.000089    -2.244145
Test-Ort: Noirmoutier-en-l'Ile | 47.000089, -2.2441449 | Datum: 2005-07-02
API-Keys zurückgegeben: []
Erste 5 Zeitstempel: []
Gefilterte Stunden für 2005-07-02 (15–18 Uhr): []


In [57]:
import requests

# Manueller Test mit den Koordinaten
url = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=47.000089&longitude=-2.2441449"
    "&start_date=2005-07-02&end_date=2005-07-02"
    "&hourly=temperature_2m,rain,wind_speed_10m"
    "&timezone=Europe%2FBerlin"
)
r = requests.get(url)
print(r.status_code)
print(r.json())

429
{'reason': 'Hourly API request limit exceeded. Please try again in the next hour.', 'error': True}


In [60]:
import pickle
import requests
import pandas as pd
import time

# --- 1. Pickle mit Koordinaten laden ---
with open("/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/cleaned_master_data_with_coordinates.pkl", "rb") as f:
    df = pickle.load(f)

# --- 2. Wetterdaten-Funktion (nur 1 Tag pro Aufruf) ---
def get_wetter(lat, lon, datum):
    if pd.isna(lat) or pd.isna(lon):
        return {}
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={datum}&end_date={datum}"
        f"&hourly=temperature_2m,rain,wind_speed_10m,"
        f"relativehumidity_2m,precipitation,winddirection_10m"
        f"&timezone=Europe%2FBerlin"
    )
    response = requests.get(url)
    if response.status_code == 200:
        return response.json().get("hourly", {})
    elif response.status_code == 429:
        print("⚠️ Rate Limit erreicht – 60 Sekunden warten...")
        time.sleep(60)
        return get_wetter(lat, lon, datum)  # nochmal versuchen
    return {}

# --- 3. Cache (jetzt mit Datum als Teil des Keys) ---
wetter_cache = {}

def wetter_fuer_ort(lat, lon, datum):
    key = (round(lat, 4), round(lon, 4), datum)
    if key not in wetter_cache:
        time.sleep(0.5)  # Pause zwischen Anfragen
        wetter_cache[key] = get_wetter(lat, lon, datum)
    return wetter_cache[key]

# --- 4. Hilfsfunktion: Mittelwert für Stundenbereich ---
def mittelwert_stundenbereich(daten, datum, stunden_von, stunden_bis, key):
    zeiten = daten.get("time", [])
    werte  = daten.get(key, [])

    gefiltert = [
        wert for zeit, wert in zip(zeiten, werte)
        if zeit.startswith(datum)
        and stunden_von <= int(zeit[11:13]) <= stunden_bis
        and wert is not None
    ]

    return round(sum(gefiltert) / len(gefiltert), 2) if gefiltert else None

# --- 5. Datum vorbereiten ---
df["date"] = pd.to_datetime(df["date"])

print(f"📅 Datumsspanne: {df['date'].min().date()} bis {df['date'].max().date()}")
print(f"📊 Anzahl Zeilen: {len(df)}")

# --- 6. Stundenbereiche je Prefix (inklusiv) ---
stunden_config = {
    "departure": (12, 15),  # 12:00 – 15:00 Uhr
    "arrival":   (15, 18),  # 15:00 – 18:00 Uhr
}

# --- 7. Wettervariablen: API-Name → Spaltenname ---
wetter_variablen = {
    "temperature_2m":      "temp_mittel",
    "rain":                "regen_mittel",
    "wind_speed_10m":      "wind_mittel",
    "relativehumidity_2m": "luftfeuchte_mittel",
    "precipitation":       "niederschlag_mittel",
    "winddirection_10m":   "windrichtung_mittel",
}

# --- 8. Wetterwerte pro Zeile eintragen ---
for prefix, (std_von, std_bis) in stunden_config.items():
    lat_col = f"{prefix}_lat"
    lon_col = f"{prefix}_lon"

    listen = {api_key: [] for api_key in wetter_variablen}

    for i, (_, row) in enumerate(df.iterrows()):
        lat   = row[lat_col]
        lon   = row[lon_col]
        datum = row["date"].strftime("%Y-%m-%d")

        # Fortschritt anzeigen
        if i % 500 == 0:
            print(f"⏳ {prefix}: Zeile {i}/{len(df)}")

        if pd.isna(lat) or pd.isna(lon):
            for api_key in wetter_variablen:
                listen[api_key].append(None)
            continue

        daten = wetter_fuer_ort(lat, lon, datum)

        for api_key in wetter_variablen:
            listen[api_key].append(
                mittelwert_stundenbereich(daten, datum, std_von, std_bis, api_key)
            )

    for api_key, spalten_suffix in wetter_variablen.items():
        df[f"{prefix}_{spalten_suffix}"] = listen[api_key]

    print(f"✅ Wetterdaten für '{prefix}' ({std_von}:00–{std_bis}:00 Uhr) hinzugefügt")

# --- 9. Speichern ---
df.to_pickle("/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/cleaned_master_data_with_coordinates_and_weather.pkl")

print(df[[
    "departure", "departure_temp_mittel", "departure_regen_mittel",
    "arrival",   "arrival_temp_mittel",   "arrival_regen_mittel"
]].head())

📅 Datumsspanne: 2005-05-08 bis 2025-09-14
📊 Anzahl Zeilen: 225692
⏳ departure: Zeile 0/225692
⏳ departure: Zeile 500/225692
⏳ departure: Zeile 1000/225692
⏳ departure: Zeile 1500/225692
⏳ departure: Zeile 2000/225692
⏳ departure: Zeile 2500/225692
⏳ departure: Zeile 3000/225692
⏳ departure: Zeile 3500/225692
⏳ departure: Zeile 4000/225692
⏳ departure: Zeile 4500/225692
⏳ departure: Zeile 5000/225692
⏳ departure: Zeile 5500/225692
⏳ departure: Zeile 6000/225692
⏳ departure: Zeile 6500/225692
⏳ departure: Zeile 7000/225692
⏳ departure: Zeile 7500/225692
⏳ departure: Zeile 8000/225692
⏳ departure: Zeile 8500/225692
⏳ departure: Zeile 9000/225692
⏳ departure: Zeile 9500/225692
⏳ departure: Zeile 10000/225692
⏳ departure: Zeile 10500/225692
⏳ departure: Zeile 11000/225692
⏳ departure: Zeile 11500/225692
⏳ departure: Zeile 12000/225692
⏳ departure: Zeile 12500/225692
⏳ departure: Zeile 13000/225692
⏳ departure: Zeile 13500/225692
⏳ departure: Zeile 14000/225692
⏳ departure: Zeile 14500/22569

KeyboardInterrupt: 

In [59]:
# Vorab prüfen: Wie viele echte API-Aufrufe werden gemacht?
unique_departure = df[["departure_lat", "departure_lon", "date"]].drop_duplicates().dropna()
unique_arrival   = df[["arrival_lat",   "arrival_lon",   "date"]].drop_duplicates().dropna()

print(f"Einzigartige departure-Anfragen: {len(unique_departure)}")
print(f"Einzigartige arrival-Anfragen:   {len(unique_arrival)}")
print(f"Gesamt API-Aufrufe:              {len(unique_departure) + len(unique_arrival)}")

Einzigartige departure-Anfragen: 1293
Einzigartige arrival-Anfragen:   1245
Gesamt API-Aufrufe:              2538


In [61]:
import pickle
import requests
import pandas as pd
import time

# --- 1. Laden ---
with open("/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/cleaned_master_data_with_coordinates.pkl", "rb") as f:
    df = pickle.load(f)

df["date"] = pd.to_datetime(df["date"])

# --- 2. Wetterdaten-Funktion ---
def get_wetter(lat, lon, datum):
    if pd.isna(lat) or pd.isna(lon):
        return {}
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={datum}&end_date={datum}"
        f"&hourly=temperature_2m,rain,wind_speed_10m,"
        f"relativehumidity_2m,precipitation,winddirection_10m"
        f"&timezone=Europe%2FBerlin"
    )
    response = requests.get(url)
    if response.status_code == 200:
        return response.json().get("hourly", {})
    elif response.status_code == 429:
        print("⚠️ Rate Limit – 3600s warten...")
        time.sleep(3600)
        response2 = requests.get(url)
        if response2.status_code == 200:
            return response2.json().get("hourly", {})
    return {}

# --- 3. Hilfsfunktion: Mittelwert für Stundenbereich ---
def mittelwert_stundenbereich(daten, datum, stunden_von, stunden_bis, key):
    zeiten = daten.get("time", [])
    werte  = daten.get(key, [])
    gefiltert = [
        wert for zeit, wert in zip(zeiten, werte)
        if zeit.startswith(datum)
        and stunden_von <= int(zeit[11:13]) <= stunden_bis
        and wert is not None
    ]
    return round(sum(gefiltert) / len(gefiltert), 2) if gefiltert else None

# --- 4. Stundenbereiche und Variablen ---
stunden_config = {
    "departure": (12, 15),
    "arrival":   (15, 18),
}

wetter_variablen = {
    "temperature_2m":      "temp_mittel",
    "rain":                "regen_mittel",
    "wind_speed_10m":      "wind_mittel",
    "relativehumidity_2m": "luftfeuchte_mittel",
    "precipitation":       "niederschlag_mittel",
    "winddirection_10m":   "windrichtung_mittel",
}

# --- 5. Nur unique Kombinationen abfragen und mergen ---
for prefix, (std_von, std_bis) in stunden_config.items():
    lat_col = f"{prefix}_lat"
    lon_col = f"{prefix}_lon"

    # Einzigartige Ort+Tag-Kombinationen
    unique = (
        df[[lat_col, lon_col, "date"]]
        .drop_duplicates()
        .dropna()
        .copy()
    )
    unique["datum_str"] = unique["date"].dt.strftime("%Y-%m-%d")

    print(f"📡 {prefix}: {len(unique)} einzigartige API-Anfragen")

    # Wetterdaten nur für unique Kombinationen abrufen
    for api_key, spalten_suffix in wetter_variablen.items():
        unique[f"{prefix}_{spalten_suffix}"] = None

    for i, (idx, row) in enumerate(unique.iterrows()):
        if i % 100 == 0:
            print(f"⏳ {prefix}: {i}/{len(unique)}")

        time.sleep(3)  # 3 Sekunden Pause → max 20 Anfragen/Minute
        daten = get_wetter(row[lat_col], row[lon_col], row["datum_str"])

        for api_key, spalten_suffix in wetter_variablen.items():
            unique.at[idx, f"{prefix}_{spalten_suffix}"] = mittelwert_stundenbereich(
                daten, row["datum_str"], std_von, std_bis, api_key
            )

    # Ergebnisse per Merge zurück in den Haupt-DataFrame
    df = df.merge(
        unique[[lat_col, lon_col, "date"] + [f"{prefix}_{s}" for s in wetter_variablen.values()]],
        on=[lat_col, lon_col, "date"],
        how="left"
    )

    print(f"✅ {prefix} fertig!")

# --- 6. Speichern ---
df.to_pickle("/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/cleaned_master_data_with_coordinates_and_weather.pkl")
print(df[[
    "departure", "departure_temp_mittel", "departure_regen_mittel",
    "departure_wind_mittel", "departure_luftfeuchte_mittel",
    "departure_niederschlag_mittel", "departure_windrichtung_mittel",
    "arrival", "arrival_temp_mittel", "arrival_regen_mittel",
    "arrival_wind_mittel", "arrival_luftfeuchte_mittel",
    "arrival_niederschlag_mittel", "arrival_windrichtung_mittel"
]].head())

📡 departure: 1293 einzigartige API-Anfragen
⏳ departure: 0/1293
⏳ departure: 100/1293
⏳ departure: 200/1293
⏳ departure: 300/1293
⏳ departure: 400/1293
⏳ departure: 500/1293
⏳ departure: 600/1293
⏳ departure: 700/1293
⏳ departure: 800/1293
⏳ departure: 900/1293
⏳ departure: 1000/1293
⏳ departure: 1100/1293
⏳ departure: 1200/1293
✅ departure fertig!
📡 arrival: 1245 einzigartige API-Anfragen
⏳ arrival: 0/1245
⏳ arrival: 100/1245
⏳ arrival: 200/1245
⏳ arrival: 300/1245
⏳ arrival: 400/1245
⏳ arrival: 500/1245
⏳ arrival: 600/1245
⏳ arrival: 700/1245
⏳ arrival: 800/1245
⏳ arrival: 900/1245
⏳ arrival: 1000/1245
⏳ arrival: 1100/1245
⏳ arrival: 1200/1245
✅ arrival fertig!
    departure departure_temp_mittel departure_regen_mittel departure_wind_mittel departure_luftfeuchte_mittel departure_niederschlag_mittel departure_windrichtung_mittel               arrival arrival_temp_mittel arrival_regen_mittel arrival_wind_mittel arrival_luftfeuchte_mittel arrival_niederschlag_mittel arrival_windrichtung

In [67]:
#print(df.head(1000))
print(df.info())
#print(df.columns)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225692 entries, 0 to 225691
Data columns (total 51 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   race                           225692 non-null  object        
 1   year                           225692 non-null  int64         
 2   url                            225692 non-null  object        
 3   rank                           225692 non-null  object        
 4   rider_url                      225692 non-null  object        
 5   time_gap                       225692 non-null  object        
 6   birthdate                      222818 non-null  object        
 7   height                         221760 non-null  float64       
 8   image_url                      182946 non-null  object        
 9   name                           222818 non-null  object        
 10  nationality                    222818 non-null  object        
 11  

In [9]:
#neuer versuch Koordinaten ohne Fehlende Werte erstellen
import pickle
import pandas as pd
from geopy.geocoders import Nominatim, Photon
from geopy.extra.rate_limiter import RateLimiter

# --- 1. Pickle laden ---
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/01_cleaned_master_data.pkl', "rb") as f:
    df = pickle.load(f)

# --- 2. Geocoder einrichten ---
nominatim = Nominatim(user_agent="mein_wetter_projekt_v2")
photon    = Photon(user_agent="mein_wetter_projekt_v2_fallback")

geocode_nominatim = RateLimiter(nominatim.geocode, min_delay_seconds=1)
geocode_photon    = RateLimiter(photon.geocode,    min_delay_seconds=1)

# Bounding Box Europa + Mittelmeer
VIEWBOX = [(-10.0, 60.0), (20.0, 35.0)]


# --- 3. Preprocessing-Funktion: Ortsnamen bereinigen ---
import re

def bereinige_ortsname(ortsname):
    """
    Versucht, aus zusammengesetzten Etappenbezeichnungen den eigentlichen
    Ortsnamen zu extrahieren, den Nominatim kennt.
    
    Strategien:
    - Klammern entfernen: "Chalet-Reynard (Mont Ventoux)" → "Mont Ventoux"
    - Bindestrich-Kombis aufsplitten: "Galibier Serre-Chevalier" → ["Galibier", "Serre-Chevalier"]
    - Bekannte Präfixe/Suffixe entfernen: "SuperBesse" → "Super Besse"
    """
    kandidaten = [ortsname]  # Original immer als ersten Versuch behalten

    # Inhalt von Klammern als eigenständiger Kandidat
    klammer_inhalt = re.findall(r'\(([^)]+)\)', ortsname)
    kandidaten += klammer_inhalt

    # Version ohne Klammern
    ohne_klammern = re.sub(r'\s*\(.*?\)', '', ortsname).strip()
    if ohne_klammern != ortsname:
        kandidaten.append(ohne_klammern)

    # Leerzeichen-getrennte Teilwörter als Kandidaten (Bsp: "Galibier Serre-Chevalier")
    teile = ortsname.split()
    if len(teile) > 1:
        kandidaten += teile           # Jedes Einzelwort
        kandidaten += [" ".join(teile[-2:]),   # Letzten 2 Wörter
                       " ".join(teile[:2])]    # Ersten 2 Wörter

    # Häufige Zusammenschreibungen normalisieren ("SuperBesse" → "Super Besse")
    normalisiert = re.sub(r'([a-z])([A-Z])', r'\1 \2', ortsname)
    if normalisiert != ortsname:
        kandidaten.append(normalisiert)

    # Duplikate entfernen, Reihenfolge beibehalten
    gesehen = set()
    return [k for k in kandidaten if k and not (k in gesehen or gesehen.add(k))]


# --- 4. Geocoding-Funktion mit automatischem Fallback ---
def get_coords(ortsname):
    """
    Stufenweise Suche:
    1. Nominatim mit Bounding Box (Europa) → verhindert falsche Kontinente
    2. Nominatim ohne Bounding Box (für Israel, Irland, Dänemark etc.)
    3. Photon als zweiter Geocoder-Dienst
    4. Alle obigen Stufen für jeden bereinigten Kandidaten-Namen wiederholen
    """
    kandidaten = bereinige_ortsname(ortsname)

    for kandidat in kandidaten:
        # Stufe 1: Nominatim mit Bounding Box
        try:
            loc = geocode_nominatim(kandidat, viewbox=VIEWBOX, bounded=True)
            if loc:
                return loc.latitude, loc.longitude
        except Exception:
            pass

        # Stufe 2: Nominatim ohne Bounding Box
        try:
            loc = geocode_nominatim(kandidat)
            if loc:
                return loc.latitude, loc.longitude
        except Exception:
            pass

        # Stufe 3: Photon
        try:
            loc = geocode_photon(kandidat)
            if loc:
                return loc.latitude, loc.longitude
        except Exception:
            pass

    return None, None


# --- 5. Koordinaten für alle einzigartigen Orte ermitteln ---
alle_orte = pd.unique(df[["departure", "arrival"]].values.ravel("K"))
coords_cache = {}

for ort in alle_orte:
    if pd.notna(ort):
        coords_cache[ort] = get_coords(ort)
        print(f"{ort}: {coords_cache[ort]}")

# --- 6. Koordinaten in neue Spalten eintragen ---
df["departure_lat"] = df["departure"].map(lambda o: coords_cache.get(o, (None, None))[0])
df["departure_lon"] = df["departure"].map(lambda o: coords_cache.get(o, (None, None))[1])
df["arrival_lat"]   = df["arrival"].map(lambda o: coords_cache.get(o, (None, None))[0])
df["arrival_lon"]   = df["arrival"].map(lambda o: coords_cache.get(o, (None, None))[1])

# --- 7. Verbliebene None-Werte ausgeben ---
print("\n--- Orte ohne Koordinaten (nach allen Fallbacks) ---")
none_orte = [ort for ort, coords in coords_cache.items() if coords == (None, None)]
for ort in none_orte:
    print(f"  ⚠ {ort}")
print(f"\nInsgesamt {len(none_orte)} Orte ohne Koordinaten.")

# --- 8. Speichern ---
df.to_pickle('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/3_Version/cleaned_master_data_with_coordinates.pkl')
#df.to_csv("../data/processed/02_cleaned_master_data_with_coordinates.csv", index=False)
print(df[["departure", "departure_lat", "departure_lon",
          "arrival",   "arrival_lat",   "arrival_lon"]].head())

Fromentine: (46.8896706, -2.1366623)
Challans: (46.8478091, -1.877431)
La Châtaigneraie: (46.65, -0.7391667)
Tours: (-4.2707569, 39.5945513)
Chambord: (48.8899454, 0.6095948)
Troyes: (48.2971626, 4.0746257)
Lunéville: (48.5916164, 6.4919563)
Pforzheim: (48.890934, 8.7025509)
Gérardmer: (48.0705772, 6.8776965)
Brignoud: (45.2625277, 5.9013413)
Courchevel: (45.4140984, 6.6349892)
Briançon: (44.8984037, 6.6436313)
Miramas: (43.582733, 5.0025809)
Agde: (43.3124696, 3.4707789)
Lézat-sur-Leze: (43.2773, 1.34623)
Mourenx: (43.3708943, -0.6294321)
Pau: (43.2957547, -0.3685668)
Albi: (12.328777, 43.165564)
Issoire: (45.5432962, 3.2501874)
Saint-Etienne: (45.4401467, 4.3873058)
Corbeil-Essonnes: (48.6137734, 2.4818087)
Strasbourg: (48.584614, 7.7507127)
Obernai: (48.4623358, 7.4817123)
Esch-sur-Alzette: (49.4959628, 5.9850306)
Huy: (50.5180856, 5.2408138)
Beauvais: (49.4300997, 2.0823355)
Lisieux: (-6.877868, 39.1773942)
Saint-Grégoire: (48.1524129, -1.685398)
Saint-Méen-le-Grand: (48.1894709, -

RateLimiter caught an error, retrying (0/2 tries). Called with (*('Mas de la Costa',), **{'viewbox': [(-10.0, 60.0), (20.0, 35.0)], 'bounded': True}).
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/opt/anaconda3/lib/python3.13/site-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
  File "/opt/anaconda3/lib/python3.13/http/client.py", line 1430, in getresponse
    response.begin()
    ~~~~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/http/client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ~~~~~~~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/http/client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/sock

Mas de la Costa: (41.2263446, 1.3480365)
Igualada: (41.5790182, 1.617346)
Cortals d’Encamp: (42.542649, 1.661098)
Santuario del Acebo: (43.1588287, -6.49896)
Alto de La Cubilla. Lena: (42.7689374, -4.4649597)
Becerril de la Sierra: (40.7163522, -3.9890699)
Plataforma de Gredos: (40.2751287, -5.23238)
Arrate: (43.1999544, -2.444046)
Lekunberri: (43.135431, -1.1418357)
La Laguna Negra de Vinuesa: (41.9975502, -2.8474113)
Sabiñanigo: (42.518364, -0.3647899)
Aramón Formigal: (42.7742017, -0.3621225)
Villanueva de Valdegovia: (42.8471449, -3.0983442)
Alto de Moncalvillo: (41.9465774, -3.1676799)
Alto de la Farrapona: (43.056861, -6.0897246)
Mirador de Ézaro: (42.9149845, -9.1050264)
Ciudad Rodrigo: (40.5974852, -6.5337192)
Alto de la Covatilla: (40.2886257, -2.1949304)
Picón Blanco: (43.1318566, -3.5474793)
Molina de Aragón: (40.8440751, -1.888874)
Alto de la Montaña de Cullera: (5.6136109, -75.5001379)
Balcón de Alicante: (38.5009795, -0.6252703)
La Manga del Mar Menor: (37.6459222, -0.717

In [10]:
df1 = pd.read_pickle('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/3_Version/cleaned_master_data_with_coordinates.pkl')

# Einstellung: Alle Zeilen der Series anzeigen (nicht kürzen)
pd.set_option('display.max_rows', None)

# Missing Values berechnen
missing_stats = df1.isnull().sum()

print("Vollständige Liste der Missing Values pro Spalte:")
print("-" * 50)
print(missing_stats)
print("-" * 50)

# Einstellung zurücksetzen (optional, falls du später wieder kurze Prints willst)
pd.reset_option('display.max_rows')

Vollständige Liste der Missing Values pro Spalte:
--------------------------------------------------
race                             0
year                             0
url                              0
rank                             0
rider_url                        0
time_gap                         0
birthdate                     2874
height                        3932
image_url                    42746
name                          2874
nationality                   2874
place_of_birth                3131
points_per_season_history     2874
season_results                2874
teams_history                 2874
weight                        3932
url_name                      2874
grand_tour_history            2895
departure                        0
arrival                          0
distance                         0
vertical_meters                  0
profile_score                    0
won_how                          0
avg_speed                        0
race_ranking            

In [12]:
df2 = pd.read_pickle('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/cleaned_master_data_with_coordinates.pkl')
# Einstellung: Alle Zeilen der Series anzeigen (nicht kürzen)
pd.set_option('display.max_rows', None)

# Missing Values berechnen
missing_stats = df2.isnull().sum()

print("Vollständige Liste der Missing Values pro Spalte:")
print("-" * 50)
print(missing_stats)
print("-" * 50)

# Einstellung zurücksetzen (optional, falls du später wieder kurze Prints willst)
pd.reset_option('display.max_rows')

Vollständige Liste der Missing Values pro Spalte:
--------------------------------------------------
race                             0
year                             0
url                              0
rank                             0
rider_url                        0
time_gap                         0
birthdate                     2874
height                        3932
image_url                    42746
name                          2874
nationality                   2874
place_of_birth                3131
points_per_season_history     2874
season_results                2874
teams_history                 2874
weight                        3932
url_name                      2874
grand_tour_history            2895
departure                        0
arrival                          0
distance                         0
vertical_meters                  0
profile_score                    0
won_how                          0
avg_speed                        0
race_ranking            

Qualitätsprüfung der Koordinaten

In [1]:
import folium
import pandas as pd
import pickle

# Pickle laden
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_coordinates_fixed_2.pkl', 'rb') as f:
    df = pickle.load(f)

# Karte zentriert auf Frankreich
m = folium.Map(location=[46.5, 2.5], zoom_start=5)

# Departure-Punkte (blau)
for _, row in df.dropna(subset=['departure_lat', 'departure_lon']).iterrows():
    folium.CircleMarker(
        location=[row['departure_lat'], row['departure_lon']],
        radius=3, color='blue', fill=True,
        popup=row['departure']
    ).add_to(m)

# Arrival-Punkte (rot)
for _, row in df.dropna(subset=['arrival_lat', 'arrival_lon']).iterrows():
    folium.CircleMarker(
        location=[row['arrival_lat'], row['arrival_lon']],
        radius=3, color='red', fill=True,
        popup=row['arrival']
    ).add_to(m)

m.save('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/koordinaten_check_2.html')

In [5]:
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/3_Version/cleaned_master_data_with_coordinates.pkl', 'rb') as f:
    df = pickle.load(f)

# Rennen finden meist in Europa statt → Grenzen prüfen
mask_invalid = (
    (df['departure_lat'] < 35) | (df['departure_lat'] > 72) |
    (df['departure_lon'] < -10) | (df['departure_lon'] > 30) |
    (df['arrival_lat'] < 35) | (df['arrival_lat'] > 72) |
    (df['arrival_lon'] < -10) | (df['arrival_lon'] > 30)
)

print("Verdächtige Einträge:")
print(df[mask_invalid][['departure', 'departure_lat', 'departure_lon',
                          'arrival', 'arrival_lat', 'arrival_lon']])

# Wie viele Zeilen hat der gesamte Datensatz?
print(f"Gesamtzeilen: {len(df)}")
print(f"Verdächtige Zeilen: {mask_invalid.sum()}")
print(f"Anteil: {mask_invalid.sum() / len(df) * 100:.1f}%")

# Welche Ortsnamen tauchen bei den Ausreißern am häufigsten auf?
ausreisser = df[mask_invalid]

print("\nTop 20 verdächtige departure-Orte:")
print(ausreisser['departure'].value_counts().head(20))

print("\nTop 20 verdächtige arrival-Orte:")
print(ausreisser['arrival'].value_counts().head(20))

# Erweiterte Grenzen für alle europäischen Radrennen
mask_invalid_v2 = (
    (df['departure_lat'] < 35) | (df['departure_lat'] > 72) |
    (df['departure_lon'] < -10) | (df['departure_lon'] > 42) |  # bis Israel
    (df['arrival_lat'] < 35) | (df['arrival_lat'] > 72) |
    (df['arrival_lon'] < -10) | (df['arrival_lon'] > 42)
)

print(f"Verdächtige Einträge mit erweiterter Box: {mask_invalid_v2.sum()}")

# NaN-Einträge separat zählen
nan_mask = (
    df['departure_lat'].isna() | df['departure_lon'].isna() |
    df['arrival_lat'].isna() | df['arrival_lon'].isna()
)

print(f"Zeilen mit fehlenden Koordinaten (NaN): {nan_mask.sum()}")
print(f"Zeilen mit tatsächlich falschen Koordinaten: {(mask_invalid & ~nan_mask).sum()}")

Verdächtige Einträge:
               departure  departure_lat  departure_lon arrival  arrival_lat  \
378     La Châtaigneraie      46.650000      -0.739167   Tours    -4.270757   
379     La Châtaigneraie      46.650000      -0.739167   Tours    -4.270757   
380     La Châtaigneraie      46.650000      -0.739167   Tours    -4.270757   
381     La Châtaigneraie      46.650000      -0.739167   Tours    -4.270757   
382     La Châtaigneraie      46.650000      -0.739167   Tours    -4.270757   
...                  ...            ...            ...     ...          ...   
225687         Alalpardo      40.629674      -3.474056  Madrid    11.593548   
225688         Alalpardo      40.629674      -3.474056  Madrid    11.593548   
225689         Alalpardo      40.629674      -3.474056  Madrid    11.593548   
225690         Alalpardo      40.629674      -3.474056  Madrid    11.593548   
225691         Alalpardo      40.629674      -3.474056  Madrid    11.593548   

        arrival_lon  
378    

In [3]:
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_coordinates_fixed_2.pkl', 'rb') as f:
    df = pickle.load(f)
    
# Genau schauen, welche Koordinaten den Ausschlag geben
print("Koordinaten-Verteilung der 'verdächtigen' Zeilen:")
print(ausreisser[['departure_lat', 'departure_lon', 
                   'arrival_lat', 'arrival_lon']].describe())

# Welche lon-Werte überschreiten 30?
print("\nOrte mit lon > 30:")
print(df[df['departure_lon'] > 30]['departure'].value_counts().head(10))
print(df[df['arrival_lon'] > 30]['arrival'].value_counts().head(10))

# Welche lat-Werte unter 35?
print("\nOrte mit lat < 35:")
print(df[df['departure_lat'] < 35]['departure'].value_counts().head(10))

Koordinaten-Verteilung der 'verdächtigen' Zeilen:


NameError: name 'ausreisser' is not defined

In [7]:
import pickle
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Pickle laden
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/3_Version/cleaned_master_data_with_coordinates.pkl', 'rb') as f:
    df = pickle.load(f)

print(f"Datensatz geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
print(df[['departure', 'departure_lat', 'departure_lon',
           'arrival', 'arrival_lat', 'arrival_lon']].head())

geolocator = Nominatim(user_agent="fix_geocoding")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Europa-Bounding Box
EUROPE_BBOX = (35, -10, 72, 42)  # (lat_min, lon_min, lat_max, lon_max)

def is_in_europe(lat, lon):
    return (35 <= lat <= 72) and (-10 <= lon <= 42)

def fix_coords(ortsname, country_hint=""):
    """Versucht Ortsname mit Länder-Hint neu zu geocodieren."""
    kandidaten = [
        f"{ortsname}, France",
        f"{ortsname}, Italy", 
        f"{ortsname}, Spain",
        f"{ortsname}, Belgium",
        f"{ortsname}, Netherlands",
        f"{ortsname}, Switzerland",
        f"{ortsname}, Germany",
        ortsname  # letzter Fallback ohne Länderzusatz
    ]
    for kandidat in kandidaten:
        try:
            loc = geocode(kandidat, viewbox=[
                (-10, 35), (42, 72)], bounded=True)
            if loc and is_in_europe(loc.latitude, loc.longitude):
                return loc.latitude, loc.longitude
        except Exception:
            continue
    return None, None

# Alle einzigartigen Ortspaare mit falschen Koordinaten finden
falsche_departure = df[
    ~df['departure_lat'].between(35, 72) | 
    ~df['departure_lon'].between(-10, 42)
][['departure', 'departure_lat', 'departure_lon']].drop_duplicates()

falsche_arrival = df[
    ~df['arrival_lat'].between(35, 72) | 
    ~df['arrival_lon'].between(-10, 42)
][['arrival', 'arrival_lat', 'arrival_lon']].drop_duplicates()

print(f"Einzigartige falsche departure-Orte: {len(falsche_departure)}")
print(f"Einzigartige falsche arrival-Orte:   {len(falsche_arrival)}")

fix_cache = {}

# Departure fixen
for _, row in falsche_departure.iterrows():
    ort = row['departure']
    if ort not in fix_cache:
        lat, lon = fix_coords(ort)
        fix_cache[ort] = (lat, lon)
        print(f"{ort}: ({lat}, {lon})")

# Arrival fixen
for _, row in falsche_arrival.iterrows():
    ort = row['arrival']
    if ort not in fix_cache:
        lat, lon = fix_coords(ort)
        fix_cache[ort] = (lat, lon)
        print(f"{ort}: ({lat}, {lon})")

# Koordinaten im DataFrame aktualisieren
def update_coords(df, fix_cache):
    for ort, (lat, lon) in fix_cache.items():
        if lat is not None:
            mask_dep = df['departure'] == ort
            df.loc[mask_dep, 'departure_lat'] = lat
            df.loc[mask_dep, 'departure_lon'] = lon

            mask_arr = df['arrival'] == ort
            df.loc[mask_arr, 'arrival_lat'] = lat
            df.loc[mask_arr, 'arrival_lon'] = lon
    return df

df = update_coords(df, fix_cache)

# Wie viele sind noch falsch?
mask_still_wrong = (
    ~df['departure_lat'].between(35, 72) |
    ~df['departure_lon'].between(-10, 42) |
    ~df['arrival_lat'].between(35, 72) |
    ~df['arrival_lon'].between(-10, 42)
)
print(f"Noch fehlerhafte Einträge: {mask_still_wrong.sum()}")

# Speichern
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_CORRECT_Coordinates.pkl', 'wb') as f:
    pickle.dump(df, f)


Datensatz geladen: 225692 Zeilen, 39 Spalten
    departure  departure_lat  departure_lon               arrival  \
0  Fromentine      46.889671      -2.136662  Noirmoutier-en-l'Ile   
1  Fromentine      46.889671      -2.136662  Noirmoutier-en-l'Ile   
2  Fromentine      46.889671      -2.136662  Noirmoutier-en-l'Ile   
3  Fromentine      46.889671      -2.136662  Noirmoutier-en-l'Ile   
4  Fromentine      46.889671      -2.136662  Noirmoutier-en-l'Ile   

   arrival_lat  arrival_lon  
0    47.000089    -2.244145  
1    47.000089    -2.244145  
2    47.000089    -2.244145  
3    47.000089    -2.244145  
4    47.000089    -2.244145  
Einzigartige falsche departure-Orte: 102
Einzigartige falsche arrival-Orte:   80
Tours: (None, None)
Albi: (None, None)
Lisieux: (None, None)
Tarbes: (39.028598, 40.6026995)
Gap: (37.9001744, 41.1338506)
London: (None, None)
Marseille: (41.6460539, 41.6482257)
Monaco: (None, None)
Barcelona: (None, None)
Rotterdam: (None, None)
Dinan: (None, None)
Macon: (No

KeyboardInterrupt: 

In [10]:
import pickle
import pandas as pd
from geopy.geocoders import Nominatim, Photon
from geopy.extra.rate_limiter import RateLimiter

# --- 1. Pickle laden ---
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/3_Version/cleaned_master_data_with_coordinates.pkl', 'rb') as f:
    df = pickle.load(f)

print(f"Datensatz geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
print(df[['departure', 'departure_lat', 'departure_lon',
           'arrival', 'arrival_lat', 'arrival_lon']].head())

# --- 2. Geocoder einrichten ---
geolocator_nom = Nominatim(user_agent="fix_geocoding_v2")
geolocator_pho = Photon(user_agent="fix_geocoding_photon")
geocode_nom = RateLimiter(geolocator_nom.geocode, min_delay_seconds=2, max_retries=3, error_wait_seconds=10)
geocode_pho = RateLimiter(geolocator_pho.geocode, min_delay_seconds=1)

# --- 3. Hilfsfunktionen ---
def is_in_europe_strict(lat, lon):
    """Prüft ob Koordinaten in Westeuropa + Mittelmeerraum liegen."""
    return (36 <= lat <= 71) and (-10 <= lon <= 28)

LAENDER = [
    "France", "Italy", "Spain", "Belgium",
    "Netherlands", "Switzerland", "Germany",
    "Portugal", "Austria", "Denmark", "Ireland"
]

def fix_coords_v2(ortsname):
    """Mehrstufige Neugeocodierung mit Länderzusatz."""
    # Stufe 1: Nominatim mit Länderzusatz (ohne bounded)
    for land in LAENDER:
        try:
            loc = geocode_nom(f"{ortsname}, {land}")
            if loc and is_in_europe_strict(loc.latitude, loc.longitude):
                return loc.latitude, loc.longitude
        except Exception:
            continue

    # Stufe 2: Nominatim ohne Länderzusatz
    try:
        loc = geocode_nom(ortsname)
        if loc and is_in_europe_strict(loc.latitude, loc.longitude):
            return loc.latitude, loc.longitude
    except Exception:
        pass

    # Stufe 3: Photon als Fallback
    try:
        loc = geocode_pho(ortsname)
        if loc and is_in_europe_strict(loc.latitude, loc.longitude):
            return loc.latitude, loc.longitude
    except Exception:
        pass

    return None, None

# --- 4. Fehlerhafte Koordinaten identifizieren ---
mask_invalid_dep = (
    ~df['departure_lat'].between(36, 71) |
    ~df['departure_lon'].between(-10, 28)
)
mask_invalid_arr = (
    ~df['arrival_lat'].between(36, 71) |
    ~df['arrival_lon'].between(-10, 28)
)

falsche_departure = df[mask_invalid_dep][['departure', 'departure_lat', 'departure_lon']].drop_duplicates()
falsche_arrival   = df[mask_invalid_arr][['arrival', 'arrival_lat', 'arrival_lon']].drop_duplicates()

print(f"\nEinzigartige falsche departure-Orte: {len(falsche_departure)}")
print(f"Einzigartige falsche arrival-Orte:   {len(falsche_arrival)}")

# --- 5. Fix-Cache aufbauen ---
fix_cache = {}

print("\n--- Starte Neugeocodierung departure ---")
for _, row in falsche_departure.iterrows():
    ort = row['departure']
    if ort not in fix_cache:
        lat, lon = fix_coords_v2(ort)
        fix_cache[ort] = (lat, lon)
        print(f"  {ort}: ({lat}, {lon})")

print("\n--- Starte Neugeocodierung arrival ---")
for _, row in falsche_arrival.iterrows():
    ort = row['arrival']
    if ort not in fix_cache:
        lat, lon = fix_coords_v2(ort)
        fix_cache[ort] = (lat, lon)
        print(f"  {ort}: ({lat}, {lon})")

# --- 6. Koordinaten im DataFrame aktualisieren ---
print("\n--- Aktualisiere DataFrame ---")
for ort, (lat, lon) in fix_cache.items():
    if lat is not None:
        mask_dep = df['departure'] == ort
        df.loc[mask_dep, 'departure_lat'] = lat
        df.loc[mask_dep, 'departure_lon'] = lon

        mask_arr = df['arrival'] == ort
        df.loc[mask_arr, 'arrival_lat'] = lat
        df.loc[mask_arr, 'arrival_lon'] = lon

# --- 7. Nicht behebbare Orte ausgeben ---
nicht_behebbar = {ort: coords for ort, coords in fix_cache.items() if coords == (None, None)}
print(f"\nOrte ohne gültige Koordinaten (manuell prüfen): {len(nicht_behebbar)}")
for ort in nicht_behebbar:
    print(f"  - {ort}")

# --- 8. Ergebnis prüfen ---
mask_noch_falsch = (
    ~df['departure_lat'].between(36, 71) |
    ~df['departure_lon'].between(-10, 28) |
    ~df['arrival_lat'].between(36, 71) |
    ~df['arrival_lon'].between(-10, 28)
)
print(f"\nNoch fehlerhafte Einträge nach Fix: {mask_noch_falsch.sum()}")

# --- 9. Speichern ---
output_path = '/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_coordinates_fixed.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(df, f)

print(f"\nGespeichert unter: {output_path}")

Datensatz geladen: 225692 Zeilen, 39 Spalten
    departure  departure_lat  departure_lon               arrival  \
0  Fromentine      46.889671      -2.136662  Noirmoutier-en-l'Ile   
1  Fromentine      46.889671      -2.136662  Noirmoutier-en-l'Ile   
2  Fromentine      46.889671      -2.136662  Noirmoutier-en-l'Ile   
3  Fromentine      46.889671      -2.136662  Noirmoutier-en-l'Ile   
4  Fromentine      46.889671      -2.136662  Noirmoutier-en-l'Ile   

   arrival_lat  arrival_lon  
0    47.000089    -2.244145  
1    47.000089    -2.244145  
2    47.000089    -2.244145  
3    47.000089    -2.244145  
4    47.000089    -2.244145  

Einzigartige falsche departure-Orte: 102
Einzigartige falsche arrival-Orte:   80

--- Starte Neugeocodierung departure ---
  Tours: (47.3900474, 0.6889268)
  Albi: (43.9277552, 2.147899)
  Lisieux: (49.1460831, 0.2255168)
  Tarbes: (43.232858, 0.0781021)
  Gap: (44.5612032, 6.0820639)
  London: (46.6295778, 5.0978317)
  Marseille: (43.2963986, 5.3777888)
  

In [13]:
import pickle
import pandas as pd
from geopy.geocoders import Nominatim, Photon
from geopy.extra.rate_limiter import RateLimiter

# --- 1. Pickle laden ---
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_coordinates_fixed.pkl', 'rb') as f:
    df = pickle.load(f)
mask_noch_falsch = (
    ~df['departure_lat'].between(36, 71) |
    ~df['departure_lon'].between(-10, 28) |
    ~df['arrival_lat'].between(36, 71) |
    ~df['arrival_lon'].between(-10, 28)
)

# Einzigartige Orte mit Koordinaten anzeigen
falsch_dep = df[mask_noch_falsch][['departure', 'departure_lat', 'departure_lon']].drop_duplicates()
falsch_arr = df[mask_noch_falsch][['arrival', 'arrival_lat', 'arrival_lon']].drop_duplicates()

print("Noch falsche departure-Orte:")
print(falsch_dep.to_string())

print("\nNoch falsche arrival-Orte:")
print(falsch_arr.to_string())

Noch falsche departure-Orte:
                       departure  departure_lat  departure_lon
132859        Base Aerea Rivolto      35.008153    -106.468106
159526                  Benasque      42.605266       0.523732
171167                     Gijón      43.544942      -5.662750
209504                   Requena      39.488078      -1.100164
213037                    Bilbao      46.685554      -1.924981
213902       ElPozo Alimentación       6.127512       1.201474
218707  Hipódromo de la Zarzuela      40.469311      -3.759068

Noch falsche arrival-Orte:
                                             arrival  arrival_lat  arrival_lon
132859                                   Piancavallo    46.109270    12.519057
159526           Estación de esquí de Ordino Arcalís    11.604186    43.152393
171167                             Alto de Cotobello   -21.244722   -42.237500
209504                 Alto de la Montaña de Cullera     5.613611   -75.500138
213037  Ascensión al Pico Jano. San Miguel d

Base Aerea Rivolto: 45.978035011385934, 13.038035110265685
Alto de la Montaña de Cullera koordinaten: 39.18480935651729, -0.2427881016637522
Madrid. Paisaje de la Luz: 40.41906563882092, -3.6921320612435906
Alto de Cotobello: 43.16526328055567, -5.6814159664845505
Ascensión al Pico Jano. San Miguel de Aguayo: 43.055960282407355, -4.023917738741224
ElPozo Alimentación: nichts gefunden --> Alhama de Murcia: 37.85042207343135, -1.4255350937479188 (dort ist das Hauptquartier von ElPozo Alimentación (Wursthersteller oder so))
ElPozo Alimentación: 37.85042207343135, -1.4255350937479188 
Estación de esquí de Ordino Arcalís: 42.64803789155921, 1.4950239096680416

El Ferial Larra Belagua: 42.98556055975256, -0.8071066240633579
Revel: 43.463378696901756, 1.9919875876890558
Polsa: 45.78417584499142, 10.94284273153374
Alberobello (Pietramadre): 40.78747242867906, 17.23849597798284
Tapia: 42.57247832933926, -4.073293448097728
Visegrád: gibt es nur in Ungarn
Balatonfüred: gibt es nur in ungarn
Kaposvár: gibt es nur in ungarn
Vlorë: gibts nur in albanien
Harrogate: ist in England
Sheffield: ist in england
Vejle: dänemark
Sønderborg: dänemark
Herning: dänemark
Horsens: dänemark
Nyborg: dänemark
Roskilde: dänemark
kopenhagen: dänemark


In [ ]:
import pickle

# Pickle laden
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_coordinates_fixed.pkl', 'rb') as f:
    df = pickle.load(f)

# Deine manuell recherchierten Koordinaten
MANUAL_FIXES = {
    "Base Aerea Rivolto":                              (45.978035,  13.038035),
    "Alto de la Montaña de Cullera":                   (39.184809,  -0.242788),
    "Madrid. Paisaje de la Luz":                       (40.419066,  -3.692132),
    "Alto de Cotobello":                               (43.165263,  -5.681416),
    "Ascensión al Pico Jano. San Miguel de Aguayo":    (43.055960,  -4.023918),
    "Estación de esquí de Ordino Arcalís":             (42.648038,   1.495024),
    "El Ferial Larra Belagua":                         (42.985561,  -0.807107),
    "Revel":                                           (43.463379,   1.991988),
    "Polsa":                                           (45.784176,  10.942843),
    "Alberobello (Pietramadre)":                       (40.787472,  17.238496),
    "Tapia":                                           (42.572478,  -4.073293),
    "ElPozo Alimentación":                             (37.850422,  -1.4255351),
    "Jerusalem":                                       (31.780308, 35.22244453),
    "Budapest":                                        (47.501046, 19.05544648),
    "London":                                          (51.518700, -0.12907561),
    "Monaco":                                          (43.739425, 7.423713845),
    "Rotterdam":                                       (51.910077, 4.476030164),
    "Leeds":                                           (53.803862, -1.54535587),
    "Bilbao":                                          (43.264569, -2.93780558),
    "Parlermo":                                        (38.117488, 13.34690671),
    "Amsterdam":                                       (52.369646, 4.895709733),
    "Bologna":                                         (44.497601, 11.43790919),
   "Torino":                                           (45.072017, 7.687880356),
   "Málaga":                                           (36.717307, -4.43905105),
   "Vigo":                                             (42.545121, -8.72731211),
   "Barcelona":                                        (41.386624, 2.146977364),
   "Milan":                                            (45.375098, 10.41007357)

}

# Koordinaten im DataFrame ersetzen
for ort, (lat, lon) in MANUAL_FIXES.items():
    df.loc[df['departure'] == ort, 'departure_lat'] = lat
    df.loc[df['departure'] == ort, 'departure_lon'] = lon
    df.loc[df['arrival'] == ort,   'arrival_lat']   = lat
    df.loc[df['arrival'] == ort,   'arrival_lon']   = lon
    print(f"  ✓ {ort}: ({lat}, {lon})")

# Prüfen ob noch fehlerhafte Einträge übrig sind
mask_check = (
    ~df['departure_lat'].between(35.5, 72) |
    ~df['departure_lon'].between(-10, 28) |
    ~df['arrival_lat'].between(35.5, 72) |
    ~df['arrival_lon'].between(-10, 28)
)
print(f"\nNoch fehlerhafte Einträge: {mask_check.sum()}")

if mask_check.sum() > 0:
    print(df[mask_check][['departure', 'departure_lat', 'departure_lon',
                           'arrival',   'arrival_lat',   'arrival_lon']].drop_duplicates().to_string())

# Speichern
output_path = '/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_coordinates_fixed_2.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(df, f)

print(f"\nGespeichert unter: {output_path}")

  ✓ Base Aerea Rivolto: (45.978035, 13.038035)
  ✓ Alto de la Montaña de Cullera: (39.184809, -0.242788)
  ✓ Madrid. Paisaje de la Luz: (40.419066, -3.692132)
  ✓ Alto de Cotobello: (43.165263, -5.681416)
  ✓ Ascensión al Pico Jano. San Miguel de Aguayo: (43.05596, -4.023918)
  ✓ Estación de esquí de Ordino Arcalís: (42.648038, 1.495024)
  ✓ El Ferial Larra Belagua: (42.985561, -0.807107)
  ✓ Revel: (43.463379, 1.991988)
  ✓ Polsa: (45.784176, 10.942843)
  ✓ Alberobello (Pietramadre): (40.787472, 17.238496)
  ✓ Tapia: (42.572478, -4.073293)
  ✓ ElPozo Alimentación: (37.850422, -1.4255351)

Noch fehlerhafte Einträge: 0

Gespeichert unter: /Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_coordinates_fixed_2.pkl


In [4]:
import pandas as pd
df2 = pd.read_pickle('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_coordinates_fixed_2.pkl')
# Einstellung: Alle Zeilen der Series anzeigen (nicht kürzen)
pd.set_option('display.max_rows', None)

# Missing Values berechnen
missing_stats = df2.isnull().sum()

print("Vollständige Liste der Missing Values pro Spalte:")
print("-" * 50)
print(missing_stats)
print("-" * 50)

# Einstellung zurücksetzen (optional, falls du später wieder kurze Prints willst)
pd.reset_option('display.max_rows')

Vollständige Liste der Missing Values pro Spalte:
--------------------------------------------------
race                             0
year                             0
url                              0
rank                             0
rider_url                        0
time_gap                         0
birthdate                     2874
height                        3932
image_url                    42746
name                          2874
nationality                   2874
place_of_birth                3131
points_per_season_history     2874
season_results                2874
teams_history                 2874
weight                        3932
url_name                      2874
grand_tour_history            2895
departure                        0
arrival                          0
distance                         0
vertical_meters                  0
profile_score                    0
won_how                          0
avg_speed                        0
race_ranking            

In [106]:
import pickle
import pandas as pd

df3 = pd.read_pickle('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_coordinates_fixed_2.pkl')

cols = ['departure', 'arrival', 'departure_lat', 'departure_lon', 'arrival_lat', 'arrival_lon']

display(df3[df3['departure'] == 'Turin'][cols].head())


,departure,arrival,departure_lat,departure_lon,arrival_lat,arrival_lon
115920,Turin,Milan,45.879305,4.249394,48.531381,0.479945
115921,Turin,Milan,45.879305,4.249394,48.531381,0.479945
115922,Turin,Milan,45.879305,4.249394,48.531381,0.479945
115923,Turin,Milan,45.879305,4.249394,48.531381,0.479945
115924,Turin,Milan,45.879305,4.249394,48.531381,0.479945


San Lorenzo al Mare giobt es im datensatz nicht.
Laías gibt es nicht im datensatz.
La Bien Aparecida ist restaurant in Madrid.

In [ ]:
import folium
import pandas as pd
import pickle

# Pickle laden
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_coordinates_fixed_2.pkl', 'rb') as f:
    df = pickle.load(f)

# Karte zentriert auf Frankreich
m = folium.Map(location=[46.5, 2.5], zoom_start=5)

# Departure-Punkte (blau)
for _, row in df.dropna(subset=['departure_lat', 'departure_lon']).iterrows():
    folium.CircleMarker(
        location=[row['departure_lat'], row['departure_lon']],
        radius=3, color='blue', fill=True,
        popup=row['departure']
    ).add_to(m)

# Arrival-Punkte (rot)
for _, row in df.dropna(subset=['arrival_lat', 'arrival_lon']).iterrows():
    folium.CircleMarker(
        location=[row['arrival_lat'], row['arrival_lon']],
        radius=3, color='red', fill=True,
        popup=row['arrival']
    ).add_to(m)

m.save('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/koordinaten_check_2.html')

In [9]:
import pickle
import pandas as pd
from geopy.geocoders import Nominatim, Photon
from geopy.extra.rate_limiter import RateLimiter

# --- 1. Pickle laden ---
with open('/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/3_Version/cleaned_master_data_with_coordinates.pkl', 'rb') as f:
    df = pickle.load(f)

print(f"Datensatz geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")

# --- 2. Geocoder mit erhöhtem Timeout einrichten ---
geolocator_nom = Nominatim(user_agent="fix_geocoding_v3", timeout=10)
geolocator_pho = Photon(user_agent="fix_geocoding_photon", timeout=10)
geocode_nom = RateLimiter(geolocator_nom.geocode, min_delay_seconds=1)
geocode_pho = RateLimiter(geolocator_pho.geocode, min_delay_seconds=1)

# --- 3. Manueller Korrektur-Cache für bekannte Problemfälle ---
# Orte, die Nominatim falsch zuordnet → direkt mit korrekten Koordinaten hinterlegen
MANUAL_FIXES = {
    "London":      (51.5074456,  -0.1277653),
    "Barcelona":   (41.3825802,   2.1770730),
    "Roma":        (41.8933203,  12.4829321),
    "Amsterdam":   (52.3730796,   4.8924534),
    "Leeds":       (53.7974185,  -1.5437941),
    "York":        (53.9656579,  -1.0743052),
    "Cambridge":   (52.2055314,   0.1186637),
    "Verona":      (45.4424977,  10.9857377),
    "Palermo":     (38.1112268,  13.3524434),
    "Napoli":      (40.8358846,  14.2487679),
    "Milano":      (45.4641943,   9.1896346),
    "Milan":       (45.4641943,   9.1896346),
    "Torino":      (45.0677551,   7.6824892),
    "Padova":      (45.3910400,  11.8058487),
    "Messina":     (38.1937571,  15.5542082),
    "Salerno":     (40.6824400,  14.7680600),
    "Assisi":      (43.0711952,  12.6146669),
    "Tivoli":      (41.9609220,  12.7988840),
    "Aosta":       (45.7373075,   7.3204025),
    "Alba":        (44.6985000,   8.0352000),
    "Bilbao":      (43.2630018,  -2.9350039),
    "Brussels":    (50.8467372,   4.3524930),
    "Rotterdam":   (51.9244424,   4.4777500),
    "Aigle":       (46.3179006,   6.9688929),
    "Monaco":      (43.7323492,   7.4276832),
    "El Pas de la Casa": (42.5427000,  1.7334000),
    "Pergola":     (43.5632097,  12.8363174),
}

# --- 4. Hilfsfunktionen ---
def is_in_europe_strict(lat, lon):
    return (36 <= lat <= 72) and (-10 <= lon <= 28)

# Länderzusätze nur für Orte ohne manuellen Fix
LAENDER = [
    "Italy", "Spain", "Belgium", "Netherlands",
    "Switzerland", "Germany", "Portugal",
    "Austria", "Denmark", "Ireland", "United Kingdom"
]

def fix_coords_v3(ortsname):
    """Erst manuellen Fix prüfen, dann mehrstufig geocodieren."""
    # Stufe 0: Manueller Fix
    if ortsname in MANUAL_FIXES:
        return MANUAL_FIXES[ortsname]

    # Stufe 1: Nominatim mit Länderzusatz
    for land in LAENDER:
        try:
            loc = geocode_nom(f"{ortsname}, {land}")
            if loc and is_in_europe_strict(loc.latitude, loc.longitude):
                return loc.latitude, loc.longitude
        except Exception:
            continue

    # Stufe 2: Nominatim ohne Länderzusatz
    try:
        loc = geocode_nom(ortsname)
        if loc and is_in_europe_strict(loc.latitude, loc.longitude):
            return loc.latitude, loc.longitude
    except Exception:
        pass

    # Stufe 3: Photon als Fallback
    try:
        loc = geocode_pho(ortsname)
        if loc and is_in_europe_strict(loc.latitude, loc.longitude):
            return loc.latitude, loc.longitude
    except Exception:
        pass

    return None, None

# --- 5. Fehlerhafte Koordinaten identifizieren ---
mask_invalid_dep = (
    ~df['departure_lat'].between(36, 71) |
    ~df['departure_lon'].between(-10, 28)
)
mask_invalid_arr = (
    ~df['arrival_lat'].between(36, 71) |
    ~df['arrival_lon'].between(-10, 28)
)

falsche_departure = df[mask_invalid_dep][['departure', 'departure_lat', 'departure_lon']].drop_duplicates()
falsche_arrival   = df[mask_invalid_arr][['arrival', 'arrival_lat', 'arrival_lon']].drop_duplicates()

print(f"Einzigartige falsche departure-Orte: {len(falsche_departure)}")
print(f"Einzigartige falsche arrival-Orte:   {len(falsche_arrival)}")

# --- 6. Fix-Cache aufbauen ---
fix_cache = {}

print("\n--- Starte Neugeocodierung departure ---")
for _, row in falsche_departure.iterrows():
    ort = row['departure']
    if ort not in fix_cache:
        lat, lon = fix_coords_v3(ort)
        fix_cache[ort] = (lat, lon)
        print(f"  {ort}: ({lat}, {lon})")

print("\n--- Starte Neugeocodierung arrival ---")
for _, row in falsche_arrival.iterrows():
    ort = row['arrival']
    if ort not in fix_cache:
        lat, lon = fix_coords_v3(ort)
        fix_cache[ort] = (lat, lon)
        print(f"  {ort}: ({lat}, {lon})")

# --- 7. Koordinaten im DataFrame aktualisieren ---
print("\n--- Aktualisiere DataFrame ---")
for ort, (lat, lon) in fix_cache.items():
    if lat is not None:
        df.loc[df['departure'] == ort, 'departure_lat'] = lat
        df.loc[df['departure'] == ort, 'departure_lon'] = lon
        df.loc[df['arrival'] == ort,   'arrival_lat']   = lat
        df.loc[df['arrival'] == ort,   'arrival_lon']   = lon

# --- 8. Nicht behebbare Orte ausgeben ---
nicht_behebbar = {ort: coords for ort, coords in fix_cache.items() if coords == (None, None)}
print(f"\nOrte ohne gültige Koordinaten (manuell prüfen): {len(nicht_behebbar)}")
for ort in nicht_behebbar:
    print(f"  - {ort}")

# --- 9. Ergebnis prüfen ---
mask_noch_falsch = (
    ~df['departure_lat'].between(36, 71) |
    ~df['departure_lon'].between(-10, 28) |
    ~df['arrival_lat'].between(36, 71) |
    ~df['arrival_lon'].between(-10, 28)
)
print(f"\nNoch fehlerhafte Einträge nach Fix: {mask_noch_falsch.sum()}")

# --- 10. Speichern ---
output_path = '/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/4_Version/cleaned_master_data_coordinates_fixed.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(df, f)

print(f"\nGespeichert unter: {output_path}")

Datensatz geladen: 225692 Zeilen, 39 Spalten
Einzigartige falsche departure-Orte: 102
Einzigartige falsche arrival-Orte:   80

--- Starte Neugeocodierung departure ---


RateLimiter caught an error, retrying (0/2 tries). Called with (*('Tours, Italy',), **{}).
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/geopy/geocoders/base.py", line 368, in _call_geocoder
    result = self.adapter.get_json(url, timeout=timeout, headers=req_headers)
  File "/opt/anaconda3/lib/python3.13/site-packages/geopy/adapters.py", line 472, in get_json
    resp = self._request(url, timeout=timeout, headers=headers)
  File "/opt/anaconda3/lib/python3.13/site-packages/geopy/adapters.py", line 500, in _request
    raise AdapterHTTPError(
    ...<4 lines>...
    )
geopy.adapters.AdapterHTTPError: Non-successful status code 429

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/geopy/extra/rate_limiter.py", line 136, in _retries_gen
    yield i  # Run the function.
    ^^^^^^^
  File "/opt/anaconda3/lib/python3.13/site-packages/geopy/ext

KeyboardInterrupt: 